# MobileNet 기반 졸음 감지 분류 프로젝트

이번 노트북은 **한 단계씩** 진행합니다.

## 주제
- 학습 중 졸음 감지 및 알림 시스템

## 전체 흐름
1. MediaPipe 좌표 기반 feature 추출 후 crop 이미지 준비
2. crop 이미지를 train / validation / test 폴더로 분할
3. 분할된 폴더를 기준으로 CNN(MobileNet) 입력 준비
4. 나중에 학습 진행

## 이번 버전에서 할 일
- 원본 이미지 경로 확인
- 클래스별 개수 확인
- train / validation / test 폴더 실제 생성
- 분할된 폴더 구조를 기준으로 DataLoader 준비

## 지금은 보류하는 것
- 실제 학습 루프
- validation 성능 비교
- test 최종 평가

즉, 이번에는 **학습 직전까지**만 준비합니다.

In [1]:
# 기본 라이브러리 import
# Path: 파일/폴더 경로를 다루기 쉽게 해줍니다.
# Counter: 클래스별 개수를 세기 쉽게 해줍니다.
# random: 데이터를 섞을 때 사용합니다.
# shutil: 파일 복사 또는 이동에 사용합니다.
from pathlib import Path
from collections import Counter
import random
import shutil
import os
import sys

# matplotlib 백엔드를 Agg로 설정하면
# 현재처럼 화면이 없는 실행 환경에서도 그래프 코드를 안전하게 돌릴 수 있습니다.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image

# 재현 가능한 결과를 위해 seed를 고정합니다.
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print('라이브러리 import 완료')
print('RANDOM_SEED =', RANDOM_SEED)

라이브러리 import 완료
RANDOM_SEED = 42


In [2]:
# 실행 위치에 따라 경로가 달라질 수 있어서 후보 경로를 2개 준비합니다.
BASE_DIR = Path.cwd()

candidate_source_dirs = [
    BASE_DIR / 'captured_images_total' / 'captured_images',
    BASE_DIR / 'y_model' / 'captured_images_total' / 'captured_images',
]

SOURCE_DATA_DIR = next((path for path in candidate_source_dirs if path.exists()), candidate_source_dirs[0])
PROJECT_DIR = SOURCE_DATA_DIR.parent
Y_MODEL_DIR = PROJECT_DIR.parent
PROJECT_ROOT_DIR = Y_MODEL_DIR.parent
SPLIT_ROOT_DIR = PROJECT_DIR / 'split_dataset'

print('현재 작업 폴더:', BASE_DIR)
print('원본 데이터 폴더:', SOURCE_DATA_DIR)
print('y_model 폴더:', Y_MODEL_DIR)
print('프로젝트 루트 폴더:', PROJECT_ROOT_DIR)
print('분할 폴더 루트:', SPLIT_ROOT_DIR)
print('원본 데이터 폴더 존재 여부:', SOURCE_DATA_DIR.exists())

현재 작업 폴더: c:\project\focus_on_class\y_model
원본 데이터 폴더: c:\project\focus_on_class\y_model\captured_images_total\captured_images
y_model 폴더: c:\project\focus_on_class\y_model
프로젝트 루트 폴더: c:\project\focus_on_class
분할 폴더 루트: c:\project\focus_on_class\y_model\captured_images_total\split_dataset
원본 데이터 폴더 존재 여부: True


In [3]:
# 클래스 폴더를 확인합니다.
# 각 폴더 이름이 분류 라벨이 됩니다.
class_dirs = sorted([path for path in SOURCE_DATA_DIR.iterdir() if path.is_dir()])
class_names = [path.name for path in class_dirs]

print('클래스 목록:')
for class_name in class_names:
    print('-', class_name)

클래스 목록:
- Attractive
- Drowsy
- Inattentive


In [4]:
# 각 이미지 파일 정보를 모읍니다.
image_extensions = {'.jpg', '.jpeg', '.png'}
image_records = []

for class_dir in class_dirs:
    for image_path in sorted(class_dir.iterdir()):
        if image_path.suffix.lower() in image_extensions:
            image_records.append({
                'label': class_dir.name,
                'path': image_path,
            })

print('전체 이미지 개수:', len(image_records))

전체 이미지 개수: 594


In [5]:
# 클래스별 이미지 개수를 확인합니다.
label_counter = Counter(record['label'] for record in image_records)

print('클래스별 이미지 개수')
for label, count in sorted(label_counter.items()):
    print(f'{label:12s}: {count}')

클래스별 이미지 개수
Attractive  : 190
Drowsy      : 224
Inattentive : 180


In [6]:
# 클래스별 샘플 이미지를 1장씩 확인합니다.
fig, axes = plt.subplots(1, len(class_names), figsize=(15, 5))

if len(class_names) == 1:
    axes = [axes]

for ax, class_name in zip(axes, class_names):
    sample_path = next(record['path'] for record in image_records if record['label'] == class_name)
    sample_image = Image.open(sample_path)
    ax.imshow(sample_image)
    ax.set_title(class_name)
    ax.axis('off')

plt.tight_layout()
plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_25456\3451524152.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1단계: train / validation / test 폴더 실제 생성

이번 단계에서는 단순히 리스트로만 나누는 것이 아니라,
실제로 아래 구조처럼 폴더를 만듭니다.

```text
split_dataset/
  train/
    Attractive/
    Drowsy/
    Inattentive/
  validation/
    Attractive/
    Drowsy/
    Inattentive/
  test/
    Attractive/
    Drowsy/
    Inattentive/
```

이 구조로 만들면 나중에 `ImageFolder`를 사용해 훨씬 쉽게 학습할 수 있습니다.

In [7]:
# train / validation / test 비율 설정
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

assert abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) < 1e-8

print('train ratio:', TRAIN_RATIO)
print('val ratio  :', VAL_RATIO)
print('test ratio :', TEST_RATIO)

train ratio: 0.7
val ratio  : 0.15
test ratio : 0.15


In [8]:
# 클래스별로 섞은 뒤, 같은 비율로 분할하는 함수입니다.
def split_records_by_class(records, train_ratio=0.7, val_ratio=0.15, seed=42):
    train_records = []
    val_records = []
    test_records = []

    labels = sorted({record['label'] for record in records})

    for label in labels:
        label_records = [record for record in records if record['label'] == label]

        rng = random.Random(seed + sum(ord(ch) for ch in label))
        rng.shuffle(label_records)

        total_count = len(label_records)
        train_end = int(total_count * train_ratio)
        val_end = train_end + int(total_count * val_ratio)

        train_records.extend(label_records[:train_end])
        val_records.extend(label_records[train_end:val_end])
        test_records.extend(label_records[val_end:])

    return train_records, val_records, test_records

In [9]:
# 실제로 분할을 수행합니다.
train_records, val_records, test_records = split_records_by_class(
    image_records,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    seed=RANDOM_SEED,
)

print('train 개수     :', len(train_records))
print('validation 개수:', len(val_records))
print('test 개수      :', len(test_records))

train 개수     : 414
validation 개수: 88
test 개수      : 92


In [10]:
# 분할 결과를 클래스별로 확인합니다.
def print_split_summary(split_name, records):
    counter = Counter(record['label'] for record in records)
    print(f'[{split_name}] 총 개수: {len(records)}')
    for label, count in sorted(counter.items()):
        print(f'  - {label:12s}: {count}')

print_split_summary('train', train_records)
print()
print_split_summary('validation', val_records)
print()
print_split_summary('test', test_records)

[train] 총 개수: 414
  - Attractive  : 133
  - Drowsy      : 156
  - Inattentive : 125

[validation] 총 개수: 88
  - Attractive  : 28
  - Drowsy      : 33
  - Inattentive : 27

[test] 총 개수: 92
  - Attractive  : 29
  - Drowsy      : 35
  - Inattentive : 28


In [11]:
# 파일을 실제 폴더로 나눌 때 사용할 모드를 설정합니다.
# 'copy' : 원본은 그대로 두고 복사본으로 split_dataset을 만듭니다. 가장 안전한 방식입니다.
# 'move' : 원본 폴더에서 파일을 빼서 split_dataset으로 이동합니다. 원본 구조가 바뀝니다.
SPLIT_MODE = 'copy'

print('현재 SPLIT_MODE =', SPLIT_MODE)

현재 SPLIT_MODE = copy


In [12]:
# split_dataset 폴더를 생성하고, 그 안에 train / validation / test / 클래스 폴더를 만듭니다.
# 이미 폴더가 있어도 다시 실행 가능하도록 exist_ok=True를 사용합니다.
split_names = ['train', 'validation', 'test']

for split_name in split_names:
    for class_name in class_names:
        target_dir = SPLIT_ROOT_DIR / split_name / class_name
        target_dir.mkdir(parents=True, exist_ok=True)

print('분할 폴더 구조 생성 완료')
print('생성 위치:', SPLIT_ROOT_DIR)

분할 폴더 구조 생성 완료
생성 위치: c:\project\focus_on_class\y_model\captured_images_total\split_dataset


In [13]:
# 파일을 실제 분할 폴더로 복사 또는 이동하는 함수입니다.
def transfer_records(records, split_name, split_root_dir, mode='copy'):
    transferred_count = 0

    for record in records:
        source_path = record['path']
        label_name = record['label']
        target_path = split_root_dir / split_name / label_name / source_path.name

        # 이미 파일이 있으면 중복 작업을 피하기 위해 건너뜁니다.
        if target_path.exists():
            continue

        if mode == 'copy':
            shutil.copy2(source_path, target_path)
        elif mode == 'move':
            shutil.move(str(source_path), str(target_path))
        else:
            raise ValueError("mode는 'copy' 또는 'move'만 가능합니다.")

        transferred_count += 1

    return transferred_count

In [14]:
# 실제로 train / validation / test 폴더에 파일을 넣습니다.
# 초보자 기준으로는 원본 보존이 중요하므로 기본값 copy를 추천합니다.
train_transferred = transfer_records(train_records, 'train', SPLIT_ROOT_DIR, mode=SPLIT_MODE)
val_transferred = transfer_records(val_records, 'validation', SPLIT_ROOT_DIR, mode=SPLIT_MODE)
test_transferred = transfer_records(test_records, 'test', SPLIT_ROOT_DIR, mode=SPLIT_MODE)

print('train 처리 파일 수     :', train_transferred)
print('validation 처리 파일 수:', val_transferred)
print('test 처리 파일 수      :', test_transferred)

train 처리 파일 수     : 0
validation 처리 파일 수: 0
test 처리 파일 수      : 0


In [15]:
# 실제 분할 폴더 안에 클래스별로 이미지가 잘 들어갔는지 확인합니다.
for split_name in split_names:
    print(f'[{split_name}]')
    for class_name in class_names:
        split_class_dir = SPLIT_ROOT_DIR / split_name / class_name
        image_count = len([path for path in split_class_dir.iterdir() if path.is_file()])
        print(f'  - {class_name:12s}: {image_count}')
    print()

[train]
  - Attractive  : 133
  - Drowsy      : 156
  - Inattentive : 125

[validation]
  - Attractive  : 28
  - Drowsy      : 33
  - Inattentive : 27

[test]
  - Attractive  : 29
  - Drowsy      : 35
  - Inattentive : 28



## 2단계: 분할된 폴더 기준으로 MobileNet 입력 준비

이제부터는 원본 폴더가 아니라 `split_dataset` 폴더를 기준으로 학습 데이터를 읽습니다.

폴더 구조가 잘 만들어졌기 때문에, 커스텀 Dataset 대신 `torchvision.datasets.ImageFolder`를 사용할 수 있습니다.
이 방법은 초보자에게 더 직관적이고 실수도 적습니다.

In [16]:
# PyTorch 관련 라이브러리를 import합니다.
import torch
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, datasets
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

print('torch version      :', torch.__version__)
print('cuda 사용 가능 여부:', torch.cuda.is_available())

torch version      : 2.11.0+cu126
cuda 사용 가능 여부: True


In [17]:
# MobileNet 입력용 이미지 크기와 transform을 정의합니다.
# train 데이터에는 augmentation을 조금 더 넣어서 과적합과 한 클래스 쏠림을 줄여봅니다.
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print('IMAGE_SIZE =', IMAGE_SIZE)

IMAGE_SIZE = 224


In [18]:
# train / validation / test 폴더 경로를 지정합니다.
TRAIN_DIR = SPLIT_ROOT_DIR / 'train'
VAL_DIR = SPLIT_ROOT_DIR / 'validation'
TEST_DIR = SPLIT_ROOT_DIR / 'test'

print('TRAIN_DIR =', TRAIN_DIR)
print('VAL_DIR   =', VAL_DIR)
print('TEST_DIR  =', TEST_DIR)

TRAIN_DIR = c:\project\focus_on_class\y_model\captured_images_total\split_dataset\train
VAL_DIR   = c:\project\focus_on_class\y_model\captured_images_total\split_dataset\validation
TEST_DIR  = c:\project\focus_on_class\y_model\captured_images_total\split_dataset\test


In [19]:
# ImageFolder는 폴더 이름을 자동으로 라벨로 인식합니다.
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=eval_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)

print('train_dataset 길이:', len(train_dataset))
print('val_dataset 길이  :', len(val_dataset))
print('test_dataset 길이 :', len(test_dataset))
print('class_to_idx      :', train_dataset.class_to_idx)

train_dataset 길이: 414
val_dataset 길이  : 88
test_dataset 길이 : 92
class_to_idx      : {'Attractive': 0, 'Drowsy': 1, 'Inattentive': 2}


In [20]:
# train 데이터 기준 클래스별 개수를 세고, class weight를 계산합니다.
# 데이터 수가 적은 클래스에 더 큰 가중치를 주면 한 클래스 쏠림을 완화하는 데 도움이 될 수 있습니다.
train_class_counts = torch.bincount(torch.tensor(train_dataset.targets), minlength=len(train_dataset.classes)).float()
class_weights = train_class_counts.sum() / (len(train_class_counts) * train_class_counts)
sample_weights = [class_weights[target].item() for target in train_dataset.targets]
train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

print('train_class_counts =', train_class_counts.tolist())
print('class_weights     =', [round(value, 4) for value in class_weights.tolist()])
print('weighted sampler 준비 완료')

train_class_counts = [133.0, 156.0, 125.0]
class_weights     = [1.0376, 0.8846, 1.104]
weighted sampler 준비 완료


In [21]:
# DataLoader를 생성합니다.
# train은 WeightedRandomSampler를 사용해서 특정 클래스가 덜 뽑히는 문제를 줄입니다.
BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print('BATCH_SIZE =', BATCH_SIZE)
print('train batch 수:', len(train_loader))
print('val batch 수  :', len(val_loader))
print('test batch 수 :', len(test_loader))

BATCH_SIZE = 16
train batch 수: 26
val batch 수  : 6
test batch 수 : 6


In [22]:
# 첫 번째 배치를 확인해서 입력 shape이 잘 나오는지 점검합니다.
sample_images, sample_labels = next(iter(train_loader))

print('이미지 배치 shape:', sample_images.shape)
print('라벨 배치 shape :', sample_labels.shape)
print('라벨 예시       :', sample_labels[:8].tolist())

이미지 배치 shape: torch.Size([16, 3, 224, 224])
라벨 배치 shape : torch.Size([16])
라벨 예시       : [2, 2, 1, 1, 1, 1, 0, 2]


In [23]:
# 학습에 사용할 장치를 선택합니다.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)

사용 장치: cuda


In [24]:
# MobileNet 모델을 생성합니다.
# pretrained weight를 사용하면 작은 데이터셋에서 더 좋은 출발점을 얻는 경우가 많습니다.
# torch 가중치 저장 폴더를 현재 프로젝트 안으로 바꿔서 권한 문제를 줄입니다.
# 다운로드가 불가능한 환경이면 자동으로 weights=None으로 돌아갑니다.
USE_PRETRAINED = True
LOCAL_TORCH_HOME = PROJECT_DIR / 'torch_cache'
LOCAL_TORCH_HOME.mkdir(parents=True, exist_ok=True)
os.environ['TORCH_HOME'] = str(LOCAL_TORCH_HOME)

try:
    selected_weights = MobileNet_V3_Small_Weights.DEFAULT if USE_PRETRAINED else None
    model = mobilenet_v3_small(weights=selected_weights)
    print('pretrained weights 사용 여부:', selected_weights is not None)
    print('TORCH_HOME =', os.environ['TORCH_HOME'])
except Exception as exc:
    print('pretrained weights를 불러오지 못해 weights=None으로 진행합니다.')
    print('원인:', exc)
    model = mobilenet_v3_small(weights=None)

# 마지막 분류기 층을 현재 클래스 개수에 맞게 수정합니다.
num_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(num_features, len(train_dataset.classes))

model = model.to(device)

print('클래스 개수:', len(train_dataset.classes))
print('마지막 분류기 층:', model.classifier[3])

pretrained weights 사용 여부: True
TORCH_HOME = c:\project\focus_on_class\y_model\captured_images_total\torch_cache
클래스 개수: 3
마지막 분류기 층: Linear(in_features=1024, out_features=3, bias=True)


## 여기까지 정리

현재까지 완료한 것:
- 원본 이미지 경로 확인
- 클래스별 이미지 개수 확인
- train / validation / test 분할 수행
- `split_dataset` 폴더 실제 생성
- 분할된 폴더 기준으로 `ImageFolder` 데이터셋 준비
- MobileNet 모델 구조 준비

현재는 **학습 직전 상태**입니다.

다음 단계에서 이어서 할 일:
1. loss function 정의
2. optimizer 정의
3. 학습 루프 작성
4. validation / test 평가

## 3단계: split_dataset 기준으로 MobileNet 학습 진행

이제 앞에서 만든 `split_dataset` 폴더를 기준으로 실제 학습 코드를 작성합니다.

이번 단계에서 할 일:
- 손실 함수(loss function) 정의
- optimizer 정의
- train 1 epoch 함수 작성
- validation / test 평가 함수 작성
- 여러 epoch 동안 학습 진행
- 가장 좋은 validation 성능의 모델로 test 평가

중요:
- train 정확도만 보면 안 됩니다.
- validation 정확도를 같이 보면서 과적합 여부를 확인해야 합니다.

In [25]:
# 학습 하이퍼파라미터를 설정합니다.
# EPOCHS: 전체 데이터를 몇 번 반복 학습할지
# LEARNING_RATE: 가중치를 얼마나 빠르게 업데이트할지
# EARLY_STOPPING_PATIENCE: validation 성능이 좋아지지 않을 때 몇 epoch까지 기다릴지
EPOCHS = 25
LEARNING_RATE = 3e-4
EARLY_STOPPING_PATIENCE = 4

print('EPOCHS =', EPOCHS)
print('LEARNING_RATE =', LEARNING_RATE)
print('EARLY_STOPPING_PATIENCE =', EARLY_STOPPING_PATIENCE)

EPOCHS = 25
LEARNING_RATE = 0.0003
EARLY_STOPPING_PATIENCE = 4


In [26]:
# 손실 함수와 optimizer를 정의합니다.
# CrossEntropyLoss에 class weight와 label smoothing을 넣어 한 클래스 과신을 줄여봅니다.
# AdamW는 weight decay를 함께 사용해서 과적합 완화에 도움을 줄 수 있습니다.
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

print('criterion =', criterion)
print('optimizer =', optimizer)

criterion = CrossEntropyLoss()
optimizer = AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0003
    maximize: False
    weight_decay: 0.0001
)


In [27]:
# 정확도를 계산하는 함수를 만듭니다.
# 가장 큰 값을 가진 인덱스를 예측 클래스로 사용합니다.
def calculate_accuracy(logits, labels):
    predictions = logits.argmax(dim=1)
    correct_count = (predictions == labels).sum().item()
    accuracy = correct_count / len(labels)
    return accuracy

In [28]:
# 1 epoch 동안 train 데이터를 사용해 모델을 학습시키는 함수입니다.
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    running_correct = 0
    total_count = 0

    for images, labels in dataloader:
        # 데이터를 현재 장치(CPU 또는 GPU)로 이동합니다.
        images = images.to(device)
        labels = labels.to(device)

        # 이전 gradient를 초기화합니다.
        optimizer.zero_grad()

        # 모델이 예측한 결과를 얻습니다.
        outputs = model(images)

        # 예측과 정답을 비교해 손실을 계산합니다.
        loss = criterion(outputs, labels)

        # 손실을 바탕으로 역전파를 수행합니다.
        loss.backward()

        # optimizer가 실제 파라미터를 업데이트합니다.
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (outputs.argmax(dim=1) == labels).sum().item()
        total_count += batch_size

    epoch_loss = running_loss / total_count
    epoch_accuracy = running_correct / total_count
    return epoch_loss, epoch_accuracy

In [29]:
# validation 또는 test처럼 평가만 할 때 사용하는 함수입니다.
# torch.no_grad()를 사용하면 gradient 계산을 하지 않아 더 가볍습니다.
def compute_classification_metrics(true_labels, pred_labels, num_classes):
    confusion = torch.zeros((num_classes, num_classes), dtype=torch.int64)

    for true_label, pred_label in zip(true_labels, pred_labels):
        confusion[int(true_label), int(pred_label)] += 1

    total = confusion.sum().item()
    correct = confusion.diag().sum().item()
    accuracy = correct / total if total else 0.0

    recall_per_class = []
    precision_per_class = []
    f1_per_class = []

    for class_index in range(num_classes):
        true_positive = confusion[class_index, class_index].item()
        false_negative = confusion[class_index, :].sum().item() - true_positive
        false_positive = confusion[:, class_index].sum().item() - true_positive

        recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) else 0.0
        precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

        recall_per_class.append(recall)
        precision_per_class.append(precision)
        f1_per_class.append(f1)

    macro_f1 = sum(f1_per_class) / num_classes if num_classes else 0.0
    balanced_accuracy = sum(recall_per_class) / num_classes if num_classes else 0.0

    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'balanced_accuracy': balanced_accuracy,
        'precision_per_class': precision_per_class,
        'recall_per_class': recall_per_class,
        'f1_per_class': f1_per_class,
        'confusion_matrix': confusion,
    }


def evaluate(model, dataloader, criterion, device, num_classes):
    model.eval()

    running_loss = 0.0
    total_count = 0
    true_labels = []
    pred_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            predictions = outputs.argmax(dim=1)

            batch_size = labels.size(0)
            running_loss += loss.item() * batch_size
            total_count += batch_size
            true_labels.extend(labels.detach().cpu().tolist())
            pred_labels.extend(predictions.detach().cpu().tolist())

    epoch_loss = running_loss / total_count
    metrics = compute_classification_metrics(true_labels, pred_labels, num_classes)
    return epoch_loss, metrics

In [30]:
# 학습 기록을 저장할 딕셔너리입니다.
# 나중에 그래프로 확인할 때 사용합니다.
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'val_macro_f1': [],
    'val_balanced_acc': [],
}

# 가장 좋은 validation accuracy를 따로 저장합니다.
best_val_accuracy = 0.0
best_val_macro_f1 = 0.0
best_val_balanced_accuracy = 0.0
best_model_state = None
epochs_without_improvement = 0

In [31]:
# epoch 단위로 학습을 진행합니다.
for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model=model,
        dataloader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
    )

    val_loss, val_metrics = evaluate(
        model=model,
        dataloader=val_loader,
        criterion=criterion,
        device=device,
        num_classes=len(train_dataset.classes),
    )

    val_acc = val_metrics['accuracy']
    val_macro_f1 = val_metrics['macro_f1']
    val_balanced_acc = val_metrics['balanced_accuracy']

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_macro_f1'].append(val_macro_f1)
    history['val_balanced_acc'].append(val_balanced_acc)

    # validation 정확도가 가장 좋았던 시점의 모델을 저장합니다.
    if (
        (val_macro_f1 > best_val_macro_f1) or
        (val_macro_f1 == best_val_macro_f1 and val_balanced_acc > best_val_balanced_accuracy) or
        (val_macro_f1 == best_val_macro_f1 and val_balanced_acc == best_val_balanced_accuracy and val_acc > best_val_accuracy)
    ):
        best_val_accuracy = val_acc
        best_val_macro_f1 = val_macro_f1
        best_val_balanced_accuracy = val_balanced_acc
        best_model_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"[Epoch {epoch + 1:02d}/{EPOCHS}] "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}, "
        f"val_macro_f1={val_macro_f1:.4f}, val_bal_acc={val_balanced_acc:.4f}"
    )

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print('early stopping 조건을 만족해서 학습을 종료합니다.')
        break

print('학습 종료')
print('최고 validation accuracy:', best_val_accuracy)
print('최고 validation macro F1:', best_val_macro_f1)
print('최고 validation balanced accuracy:', best_val_balanced_accuracy)

[Epoch 01/25] train_loss=0.9361, train_acc=0.5870, val_loss=1.2768, val_acc=0.4318, val_macro_f1=0.3430, val_bal_acc=0.4366
[Epoch 02/25] train_loss=0.6735, train_acc=0.7923, val_loss=1.1927, val_acc=0.5227, val_macro_f1=0.4809, val_bal_acc=0.5264
[Epoch 03/25] train_loss=0.6080, train_acc=0.8164, val_loss=0.8875, val_acc=0.6136, val_macro_f1=0.5957, val_bal_acc=0.6251
[Epoch 04/25] train_loss=0.4911, train_acc=0.8986, val_loss=0.8270, val_acc=0.6591, val_macro_f1=0.6424, val_bal_acc=0.6593
[Epoch 05/25] train_loss=0.4799, train_acc=0.9058, val_loss=0.9658, val_acc=0.5795, val_macro_f1=0.5539, val_bal_acc=0.5859
[Epoch 06/25] train_loss=0.4497, train_acc=0.9130, val_loss=1.0784, val_acc=0.6705, val_macro_f1=0.6662, val_bal_acc=0.6944
[Epoch 07/25] train_loss=0.4477, train_acc=0.9348, val_loss=0.7602, val_acc=0.7045, val_macro_f1=0.6870, val_bal_acc=0.6965
[Epoch 08/25] train_loss=0.4039, train_acc=0.9734, val_loss=0.9181, val_acc=0.6136, val_macro_f1=0.5568, val_bal_acc=0.6072
[Epoch 0

In [32]:
# validation 성능이 가장 좋았던 모델 상태를 다시 불러옵니다.
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    model = model.to(device)
    print('best validation model 불러오기 완료')
else:
    print('저장된 best model이 없습니다. 현재 model을 그대로 사용합니다.')

best validation model 불러오기 완료


In [33]:
# test 데이터로 최종 성능을 평가합니다.
test_loss, test_metrics = evaluate(
    model=model,
    dataloader=test_loader,
    criterion=criterion,
    device=device,
    num_classes=len(train_dataset.classes),
)

test_acc = test_metrics['accuracy']
test_macro_f1 = test_metrics['macro_f1']
test_balanced_acc = test_metrics['balanced_accuracy']

print(f'test_loss = {test_loss:.4f}')
print(f'test_acc  = {test_acc:.4f}')
print(f'test_macro_f1 = {test_macro_f1:.4f}')
print(f'test_balanced_acc = {test_balanced_acc:.4f}')

test_loss = 0.7171
test_acc  = 0.7500
test_macro_f1 = 0.7369
test_balanced_acc = 0.7476


In [34]:
# test 샘플 몇 장에 대해 실제 라벨과 예측 라벨을 직접 확인합니다.
# 어떤 식으로 틀리는지 눈으로 보는 데 도움이 됩니다.
def denormalize_image(image_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    image = image_tensor.cpu() * std + mean
    image = image.clamp(0, 1)
    return image.permute(1, 2, 0).numpy()

num_samples_to_show = min(6, len(test_dataset))
sample_indices = list(range(num_samples_to_show))

plt.figure(figsize=(15, 8))

for plot_idx, sample_idx in enumerate(sample_indices, start=1):
    image_tensor, true_label = test_dataset[sample_idx]
    model.eval()
    with torch.no_grad():
        output = model(image_tensor.unsqueeze(0).to(device))
        pred_label = output.argmax(dim=1).item()

    plt.subplot(2, 3, plot_idx)
    plt.imshow(denormalize_image(image_tensor))
    plt.title(f'True: {train_dataset.classes[true_label]}\nPred: {train_dataset.classes[pred_label]}')
    plt.axis('off')

plt.tight_layout()
plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_25456\2814579850.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [35]:
# test 데이터 기준 confusion matrix와 클래스별 세부 지표를 확인합니다.
# 행은 실제 클래스, 열은 예측 클래스를 의미합니다.
confusion_matrix = test_metrics['confusion_matrix']

print(confusion_matrix)
print('class_names =', train_dataset.classes)

for class_name, precision, recall, f1 in zip(
    train_dataset.classes,
    test_metrics['precision_per_class'],
    test_metrics['recall_per_class'],
    test_metrics['f1_per_class'],
):
    print(
        f'{class_name:12s} | precision={precision:.4f} | recall={recall:.4f} | f1={f1:.4f}'
    )

tensor([[29,  0,  0],
        [ 6, 26,  3],
        [12,  2, 14]])
class_names = ['Attractive', 'Drowsy', 'Inattentive']
Attractive   | precision=0.6170 | recall=1.0000 | f1=0.7632
Drowsy       | precision=0.9286 | recall=0.7429 | f1=0.8254
Inattentive  | precision=0.8235 | recall=0.5000 | f1=0.6222


In [36]:
# confusion matrix를 그래프로 표시합니다.
plt.figure(figsize=(6, 5))
plt.imshow(confusion_matrix.numpy(), cmap='Blues')
plt.title('Confusion Matrix')
plt.colorbar()
plt.xticks(range(len(train_dataset.classes)), train_dataset.classes, rotation=45)
plt.yticks(range(len(train_dataset.classes)), train_dataset.classes)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')

for i in range(confusion_matrix.shape[0]):
    for j in range(confusion_matrix.shape[1]):
        plt.text(j, i, int(confusion_matrix[i, j]), ha='center', va='center', color='black')

plt.tight_layout()
plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_25456\3881962500.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [37]:
# train / validation 성능 변화를 그래프로 확인합니다.
# early stopping이 동작할 수 있으므로 실제 기록 길이에 맞춰 x축을 만듭니다.
epochs_range = range(1, len(history['train_loss']) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history['train_loss'], label='train_loss')
plt.plot(epochs_range, history['val_loss'], label='val_loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history['train_acc'], label='train_acc')
plt.plot(epochs_range, history['val_acc'], label='val_acc')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_25456\3837888148.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 최종 정리

현재 노트북은 아래 흐름으로 끝까지 진행할 수 있습니다.

1. 원본 이미지 확인
2. train / validation / test 폴더 실제 생성
3. `ImageFolder`로 데이터셋 준비
4. MobileNet 모델 준비
5. 학습 진행
6. validation 확인
7. test 최종 평가

이번 버전에서 추가한 개선 포인트:
- confusion matrix 추가
- early stopping 추가
- pretrained weights 사용 옵션 추가
- augmentation 강화
- AdamW 적용
- class weights 적용
- test 예측 샘플 시각화 추가

다음에 이어서 해볼 수 있는 개선 포인트:
- MediaPipe crop 개선
- 실시간 알림 시스템용 inference 코드 연결

## 4단계: 학습된 모델 저장

여기서는 지금까지 학습한 MobileNet 모델을 파일로 저장합니다.

저장해두면 나중에 OpenCV 실시간 분류 코드에서 다시 불러와 사용할 수 있습니다.

이번 프로젝트에서는 두 가지 형식으로 저장합니다.
- `.pth`
- `.pt`

초보자 기준에서는 둘 다 비슷하게 생각해도 괜찮습니다.
지금은 나중에 불러오기 편하게 체크포인트 딕셔너리 형태로 저장합니다.

In [38]:
# 모델 저장 경로를 설정합니다.
# 실행 위치와 상관없이 실제 y_model 폴더에 저장되도록 고정합니다.
MODEL_PTH_PATH = Y_MODEL_DIR / 'mobilenet_drowsiness.pth'
MODEL_PT_PATH = Y_MODEL_DIR / 'mobilenet_drowsiness.pt'

print('MODEL_PTH_PATH =', MODEL_PTH_PATH)
print('MODEL_PT_PATH  =', MODEL_PT_PATH)

MODEL_PTH_PATH = c:\project\focus_on_class\y_model\mobilenet_drowsiness.pth
MODEL_PT_PATH  = c:\project\focus_on_class\y_model\mobilenet_drowsiness.pt


In [39]:
# 나중에 다시 불러올 때 필요한 정보를 딕셔너리로 묶습니다.
checkpoint = {
    'model_state_dict': model.state_dict(),
    'class_names': train_dataset.classes,
    'image_size': IMAGE_SIZE,
    'normalize_mean': [0.485, 0.456, 0.406],
    'normalize_std': [0.229, 0.224, 0.225],
    'best_val_acc': best_val_accuracy,
    'best_val_macro_f1': best_val_macro_f1,
    'best_val_balanced_acc': best_val_balanced_accuracy,
    'test_acc': test_acc,
    'test_macro_f1': test_macro_f1,
    'test_balanced_acc': test_balanced_acc,
}

print('checkpoint 준비 완료')
print('class_names =', checkpoint['class_names'])

checkpoint 준비 완료
class_names = ['Attractive', 'Drowsy', 'Inattentive']


In [40]:
# .pth 파일로 저장합니다.
torch.save(checkpoint, MODEL_PTH_PATH)
print('.pth 저장 완료:', MODEL_PTH_PATH)

.pth 저장 완료: c:\project\focus_on_class\y_model\mobilenet_drowsiness.pth


In [41]:
# .pt 파일로도 저장합니다.
torch.save(checkpoint, MODEL_PT_PATH)
print('.pt 저장 완료:', MODEL_PT_PATH)

.pt 저장 완료: c:\project\focus_on_class\y_model\mobilenet_drowsiness.pt


## 5단계: 정확도 개선 포인트 정리

현재 결과에서 `Drowsy`, `Inattentive` 쏠림이 심하고 정확도가 낮다면,
이 노트북에서 먼저 확인해야 할 핵심 포인트는 아래와 같습니다.

1. `WeightedRandomSampler`가 적용된 상태로 다시 학습했는지
2. `label smoothing`과 `class weight`가 같이 적용됐는지
3. `best_val_accuracy`, `test_acc`, `confusion matrix`가 실제로 개선됐는지
4. 여전히 한 클래스에 몰리면 데이터 자체를 다시 점검해야 하는지

중요:
- 지금 모델 정확도가 낮다면 실시간 결과도 그대로 낮게 나옵니다.
- 실시간 코드보다 먼저 학습 성능을 올리는 것이 우선입니다.

In [42]:
# 현재 학습 결과를 한 번 더 요약해서 봅니다.
print('best_val_accuracy      =', best_val_accuracy)
print('best_val_macro_f1      =', best_val_macro_f1)
print('best_val_balanced_acc  =', best_val_balanced_accuracy)
print('test_acc               =', test_acc)
print('test_macro_f1          =', test_macro_f1)
print('test_balanced_acc      =', test_balanced_acc)
print('class_names            =', train_dataset.classes)

if test_macro_f1 < 0.6 or test_balanced_acc < 0.6:
    print('현재 정확도는 아직 낮은 편입니다. 실시간 분류 결과를 그대로 믿기 어렵습니다.')
else:
    print('기본적인 분류 성능은 확보된 상태입니다.')

best_val_accuracy      = 0.8181818181818182
best_val_macro_f1      = 0.810698062482618
best_val_balanced_acc  = 0.8159371492704827
test_acc               = 0.75
test_macro_f1          = 0.7369256474519633
test_balanced_acc      = 0.7476190476190476
class_names            = ['Attractive', 'Drowsy', 'Inattentive']
기본적인 분류 성능은 확보된 상태입니다.


## 6단계: 헷갈리는 샘플 점검

실시간 분류 품질을 올리려면 특정 클래스를 억지로 높이기보다, 어떤 샘플이 왜 헷갈리는지 먼저 보는 편이 더 효과적입니다.

아래 셀은 `test` 데이터에서 오분류된 이미지 일부를 직접 보여줍니다.
특히 `Drowsy`와 `Inattentive`가 서로 섞이는 장면을 먼저 점검해보세요.

라벨 기준은 [docs/labeling_guidelines.md](C:/project/focus_on_class/docs/labeling_guidelines.md)를 같이 참고하면 좋습니다.

In [43]:
# test 데이터에서 오분류 샘플을 모아 직접 확인합니다.
# limit_per_pair 값을 늘리면 같은 클래스 쌍의 예시를 더 많이 볼 수 있습니다.
def collect_misclassified_samples(model, dataset, class_names, device, limit_per_pair=6):
    model.eval()
    misclassified = []
    pair_counts = {}

    with torch.no_grad():
        for sample_index in range(len(dataset)):
            image_tensor, true_label = dataset[sample_index]
            input_tensor = image_tensor.unsqueeze(0).to(device)
            outputs = model(input_tensor)
            probabilities = torch.softmax(outputs, dim=1)
            confidence, pred_label = probabilities.max(dim=1)

            true_index = int(true_label)
            pred_index = int(pred_label.item())

            if true_index == pred_index:
                continue

            pair_key = (class_names[true_index], class_names[pred_index])
            pair_counts.setdefault(pair_key, 0)

            if pair_counts[pair_key] >= limit_per_pair:
                continue

            pair_counts[pair_key] += 1
            misclassified.append({
                'sample_index': sample_index,
                'true_label': class_names[true_index],
                'pred_label': class_names[pred_index],
                'confidence': float(confidence.item()),
                'image_tensor': image_tensor.cpu(),
                'image_path': dataset.samples[sample_index][0],
            })

    return misclassified


def show_misclassified_samples(samples, max_samples=12):
    samples_to_show = samples[:max_samples]

    if not samples_to_show:
        print('표시할 오분류 샘플이 없습니다.')
        return

    columns = 3
    rows = (len(samples_to_show) + columns - 1) // columns
    plt.figure(figsize=(15, 4 * rows))

    for plot_index, sample in enumerate(samples_to_show, start=1):
        plt.subplot(rows, columns, plot_index)
        plt.imshow(denormalize_image(sample['image_tensor']))
        plt.title(
            f"True: {sample['true_label']}\nPred: {sample['pred_label']} ({sample['confidence']:.2f})"
        )
        plt.axis('off')

    plt.tight_layout()
    plt.show()

    print('오분류 샘플 경로:')
    for sample in samples_to_show:
        print(f"- [{sample['true_label']} -> {sample['pred_label']}] {sample['image_path']}")


misclassified_samples = collect_misclassified_samples(
    model=model,
    dataset=test_dataset,
    class_names=train_dataset.classes,
    device=device,
    limit_per_pair=6,
)

print('수집한 오분류 샘플 수 =', len(misclassified_samples))
show_misclassified_samples(misclassified_samples, max_samples=12)

수집한 오분류 샘플 수 = 17
오분류 샘플 경로:
- [Drowsy -> Attractive] c:\project\focus_on_class\y_model\captured_images_total\split_dataset\test\Drowsy\drowsy_20260423_152308_004924_001.jpg
- [Drowsy -> Inattentive] c:\project\focus_on_class\y_model\captured_images_total\split_dataset\test\Drowsy\drowsy_20260423_162549_281900_010.jpg
- [Drowsy -> Attractive] c:\project\focus_on_class\y_model\captured_images_total\split_dataset\test\Drowsy\drowsy_20260423_162549_281900_015.jpg
- [Drowsy -> Attractive] c:\project\focus_on_class\y_model\captured_images_total\split_dataset\test\Drowsy\drowsy_20260423_162748_045875_013.jpg
- [Drowsy -> Inattentive] c:\project\focus_on_class\y_model\captured_images_total\split_dataset\test\Drowsy\drowsy_20260423_162748_150995_002.jpg
- [Drowsy -> Inattentive] c:\project\focus_on_class\y_model\captured_images_total\split_dataset\test\Drowsy\drowsy_20260423_162748_150995_004.jpg
- [Drowsy -> Attractive] c:\project\focus_on_class\y_model\captured_images_total\split_dataset\tes

C:\Users\Admin\AppData\Local\Temp\ipykernel_25456\1697763601.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 실시간 OpenCV 연결

아래 마지막 코드 셀을 실행하면 `main.py`의 실시간 분류가 바로 시작됩니다.

주의:
- 현재 모델 정확도가 낮으면 실시간 결과도 그대로 부정확할 수 있습니다.
- 그래도 연결 자체는 바로 확인할 수 있도록 구성합니다.

In [44]:
# 이 셀 하나로 체크포인트 확인, import, 옵션 설정, 실시간 실행까지 한 번에 진행합니다.
REALTIME_PT_PATH = Y_MODEL_DIR / 'mobilenet_drowsiness.pt'
REALTIME_PTH_PATH = Y_MODEL_DIR / 'mobilenet_drowsiness.pth'
REALTIME_FACE_MODEL_PATH = PROJECT_ROOT_DIR / 'mp_model' / 'face_landmarker.task'

print('REALTIME_PT_PATH exists :', REALTIME_PT_PATH.exists())
print('REALTIME_PTH_PATH exists:', REALTIME_PTH_PATH.exists())
print('FACE_MODEL exists       :', REALTIME_FACE_MODEL_PATH.exists())

if str(PROJECT_ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT_DIR))

from main import run_realtime

camera_index = 0
drowsy_threshold_frames = 15
min_confidence = 0.40

run_realtime(
    camera_index=camera_index,
    drowsy_threshold_frames=drowsy_threshold_frames,
    min_confidence=min_confidence,
)

REALTIME_PT_PATH exists : True
REALTIME_PTH_PATH exists: True
FACE_MODEL exists       : True
실시간 분류를 시작합니다.
사용 체크포인트: C:\project\focus_on_class\y_model\mobilenet_drowsiness.pt
디버그 crop 저장 폴더: C:\project\focus_on_class\y_model\realtime_debug_crops
종료하려면 화면에서 'q' 키를 누르세요.
현재 화면에는 집중도 퍼센트 대신 예측 클래스와 confidence만 표시합니다.
